# Run all M3 notebooks on Kaggle in one session

On Kaggle every notebook has its own `/kaggle/working` folder, so notebooks 5 to 13 are run from here,
one after another, in the same session: later notebooks can then use earlier results (13 collects
them all). Each notebook is saved with its outputs in `/kaggle/working/executed/`, and every result in
`/kaggle/working/m3_results.zip`.

1. *File > Import notebook*: this file.
2. *Add input*: your private dataset `veristamp-m3-inputs` (layout in `m3_experiments/README.md`).
3. *Settings*: Accelerator **GPU T4 x2**, Internet **on**.
4. Set the switches below, then *Save Version > Save & Run All (Commit)*. It runs in the background
   (about 2 to 3 hours with inpainting on 200 photos), and the outputs stay with that version.

A failed notebook does not stop the others; the last cell lists which ones failed.

In [1]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

Code found at:  /kaggle/input/veristamp-m3-inputs/veristamp-m3-inputs (3)/code/VeriStamp/m3_experiments/settings.py
Model found at: /kaggle/input/veristamp-m3-inputs/veristamp-m3-inputs (3)/m3/best.pt
  m3/best.pt             found
  m3/test                not in the inputs
  m3/test_manifest.csv   found
  genimage/splits        found
  genimage/images        found
  m1/best.pt             found
  m1/vslib_m1_final      found
Working in /kaggle/working/VeriStamp/m3_experiments


## Switches

In [2]:
RUN = ['5_setup_check', '6_rq2_screenshot', '7_image_quality', '8_rq2_vae_roundtrip',
       '9_rq2_inpainting', '10_genimage_check', '11_rq3_detectors', '12_pipeline', '13_report_tables']
os.environ['M3E_USE_M1'] = '1'
os.environ['M3E_USE_M2'] = '0'
os.environ['M3E_USE_FUSION'] = '0'

## Run

In [3]:
import time
out_dir = Path('/kaggle/working/executed')
out_dir.mkdir(parents=True, exist_ok=True)
status = {}
for name in RUN:
    start = time.time()
    r = subprocess.run(['jupyter', 'nbconvert', '--to', 'notebook', '--execute', f'{name}.ipynb',
                        '--ExecutePreprocessor.timeout=-1', '--ExecutePreprocessor.kernel_name=python3',
                        '--output-dir', str(out_dir), '--output', name], capture_output=True, text=True)
    status[name] = 'ok' if r.returncode == 0 else 'FAILED'
    print(f'{name}: {status[name]} ({(time.time() - start) / 60:.1f} min)')
    if r.returncode:
        print(r.stderr[-3000:])

5_setup_check: ok (4.7 min)
6_rq2_screenshot: ok (11.9 min)
7_image_quality: ok (1.9 min)
8_rq2_vae_roundtrip: ok (30.5 min)
9_rq2_inpainting: ok (56.3 min)
10_genimage_check: ok (4.4 min)
11_rq3_detectors: ok (2.7 min)
12_pipeline: ok (12.4 min)
13_report_tables: ok (0.1 min)


In [4]:
import shutil
print(shutil.make_archive('/kaggle/working/m3_results', 'zip', 'results'))
print('Executed notebooks with outputs:', out_dir)
failed = [n for n, s in status.items() if s != 'ok']
print('All notebooks ran.' if not failed else f'Failed: {failed} (open them in {out_dir} to see the error)')

/kaggle/working/m3_results.zip
Executed notebooks with outputs: /kaggle/working/executed
All notebooks ran.


In [5]:
!grep -o "SETUP OK[^\"]*\|MISMATCH[^\"]*" /kaggle/working/executed/5_setup_check.ipynb
!cat results/rq2_screenshot_summary.md results/image_quality_summary.md results/rq2_vae_summary.md

SETUP OK: results match notebook 4.\n
SETUP OK: results match notebook 4.' if ok else '\\nMISMATCH: stop and tell Rishitha.')
| edit | n_photos | bit_accuracy | valid_verdict | invalidation_rate | all_32_bits_correct | wrong_but_valid |
|---|---|---|---|---|---|---|
| No edit (PNG) | 500 | 99.86% | 96.80% | 3.20% | 96.80% | 0 |
| Resize to 80% only | 500 | 99.86% | 96.60% | 3.40% | 96.60% | 0 |
| Resize to 80% and back | 500 | 99.83% | 95.80% | 4.20% | 95.80% | 0 |
| Gaussian blur r=0.5 only | 500 | 99.86% | 96.80% | 3.20% | 96.80% | 0 |
| Brightness/contrast +-5% only | 500 | 99.79% | 95.80% | 4.20% | 95.80% | 0 |
| Screenshot simulation (all steps) | 500 | 99.71% | 94.20% | 5.80% | 94.20% | 0 |
| Screenshot simulation + JPEG 90 | 500 | 99.56% | 90.60% | 9.40% | 90.60% | 0 |
| metric | mean | std | median | worst |
|---|---|---|---|---|
| PSNR (dB) | 31.5323 | 1.6853 | 31.5842 | 25.8631 |
| SSIM | 0.9428 | 0.0295 | 0.9479 | 0.6895 |
| LPIPS (AlexNet) | 0.0737 | 0.0257 | 0.0696 | 0.234